# OpenScope Community Predictive Processing — Analysis Demo

This notebook demonstrates two core analyses from the **openscope-pp** package:

| Section | Dataset | What we show |
|---|---|---|
| **1 — RF Mapping** | Mesoscope (dandiset 001768, sub-839909) | Population PSTH + per-ROI receptive field maps for VISp plane 0 |
| **2 — Oddball Mismatch** | Ecephys (dandiset 001256) | Visual SUA responses to standard vs deviant stimuli, split by deviant type |

All data stream directly from DANDI — nothing is downloaded locally.

> **Runtime estimate:** ~4 min on a standard Colab CPU instance.

In [ ]:
# Install dependencies (run once per session)
!pip install -q git+https://github.com/maierav/claupenscope.git \
    dandi remfile h5py pynwb scipy matplotlib numpy pandas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
import h5py, remfile, time

from openscope_pp.loaders.streaming import open_nwb
from openscope_pp.loaders.trials import load_trials

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("Imports OK")

---
## Section 1 — Mesoscope Receptive Field Mapping

**Dataset:** dandiset 001768, sub-839909 (3.6 GB NWB, streamed via remfile)  
**Stimuli:** 9 × 9 spatial grid (±40°), 15 trials/position, 1215 trials total  
**Analysis plane:** VISp_0 — 350 soma-only ROIs at ~10.7 Hz (GCaMP)

Steps:
1. Stream the NWB file and load RF trial times + grid positions
2. Bulk-read the dFF span covering all trials; interpolate per-ROI snippets
3. Z-score, average over trials per grid position → RF map per ROI
4. Plot population PSTH (alignment check) and top individual RF maps

In [ ]:
# ── Config ────────────────────────────────────────────────────────────
MESO_ASSET_ID = "00451971-f839-4cb0-a11b-1d2e12c8ad6b"   # sub-839909
PLANE         = "VISp_0"
WINDOW        = (-0.5, 1.5)     # s around stimulus onset
RESP_WIN      = (0.1, 0.8)      # response scoring window (slow GCaMP dynamics)
RF_SMOOTH     = 0.8             # gaussian sigma in grid units
N_TOP         = 12              # individual ROI maps to show

# ── Open streaming NWB ────────────────────────────────────────────────
print("Connecting to DANDI…")
t0 = time.time()
handle = open_nwb(MESO_ASSET_ID)
h5 = handle.h5
print(f"  Opened in {time.time()-t0:.1f}s  (technique: {handle.technique})")

# ── Load RF trial table ───────────────────────────────────────────────
def decode_col(arr):
    return np.array([float(v.decode() if isinstance(v, bytes) else v) for v in arr])

rf_grp  = h5["intervals"]["RF mapping_presentations"]
starts  = rf_grp["start_time"][:]
xs_raw  = decode_col(rf_grp["X"][:])
ys_raw  = decode_col(rf_grp["Y"][:])
xs_grid = np.array(sorted(np.unique(xs_raw)))
ys_grid = np.array(sorted(np.unique(ys_raw)))

print(f"RF mapping: {len(starts)} trials, {len(xs_grid)}×{len(ys_grid)} grid, "
      f"{len(starts)/(len(xs_grid)*len(ys_grid)):.0f} trials/position")
print(f"  X: {xs_grid}°")
print(f"  Y: {ys_grid}°")

# ── Load dFF + soma mask for chosen plane ─────────────────────────────
base     = f"processing/{PLANE}"
dff_path = f"{base}/dff_timeseries/dff_timeseries"
ts       = h5[f"{dff_path}/timestamps"][:]
data     = h5[f"{dff_path}/data"]            # lazy HDF5 dataset
is_soma  = h5[f"{base}/image_segmentation/roi_table/is_soma"][:].astype(bool)
n_soma   = is_soma.sum()
print(f"\n{PLANE}: {data.shape[1]} ROIs → {n_soma} soma  |  rate: "
      f"{1/float(np.median(np.diff(ts[len(ts)//2:len(ts)//2+200]))):.1f} Hz")

In [ ]:
# ── Extract snippets via np.interp (robust to gaps / clock drift) ─────
valid   = (starts + WINDOW[0] >= ts[0]) & (starts + WINDOW[1] <= ts[-1])
onsets  = starts[valid]
xs_v    = xs_raw[valid]
ys_v    = ys_raw[valid]
print(f"{valid.sum()}/{len(starts)} RF trials within recording window")

dt       = float(np.median(np.diff(ts[len(ts)//2 : len(ts)//2 + 200])))
n_samp   = int(round((WINDOW[1] - WINDOW[0]) / dt))
t_rel    = np.linspace(WINDOW[0], WINDOW[0] + (n_samp - 1) * dt, n_samp)
tc       = t_rel + dt / 2

# Bulk HDF5 read covering the full RF block
t0_span = onsets.min() + WINDOW[0] - 2.0
t1_span = onsets.max() + WINDOW[1] + 2.0
i0 = max(0, int(np.searchsorted(ts, t0_span)))
i1 = min(data.shape[0], int(np.searchsorted(ts, t1_span)) + 1)
print(f"Reading HDF5 span [{i0}:{i1}] = {i1-i0} samples × {data.shape[1]} ROIs …")
trace      = data[i0:i1, :].astype(np.float32)
ts_span    = ts[i0:i1]
trace_soma = trace[:, is_soma]   # keep soma only

# Interpolate: (n_trials, n_soma, n_samples)
t_query = onsets[:, None] + t_rel[None, :]   # (n_trials, n_samp)
t_flat  = t_query.ravel()
snip    = np.full((len(onsets), n_soma, n_samp), np.nan, dtype=np.float32)
for roi in range(n_soma):
    y   = trace_soma[:, roi]
    fin = np.isfinite(y)
    if fin.sum() < 10: continue
    vals = np.interp(t_flat, ts_span[fin], y[fin], left=np.nan, right=np.nan)
    snip[:, roi, :] = vals.reshape(len(onsets), n_samp)
print(f"Snippets shape: {snip.shape}")

# Z-score per ROI using pre-stimulus baseline
bl_mask = (tc >= -0.5) & (tc < 0.0)
mu  = np.nanmean(snip[:, :, bl_mask], axis=(0, 2), keepdims=True)
sig = np.nanstd( snip[:, :, bl_mask], axis=(0, 2), keepdims=True)
sig = np.where(sig > 1e-6, sig, 1.0)
z   = (snip - mu) / sig

# Build RF maps: (n_soma, n_y, n_x)
resp_mask = (tc >= RESP_WIN[0]) & (tc < RESP_WIN[1])
rf_maps   = np.zeros((n_soma, len(ys_grid), len(xs_grid)), dtype=np.float32)
for xi, x in enumerate(xs_grid):
    for yi, y in enumerate(ys_grid):
        m = (xs_v == x) & (ys_v == y)
        if m.sum() == 0: continue
        resp = np.nanmean(z[m][:, :, resp_mask], axis=(0, 2))
        base = np.nanmean(z[m][:, :, bl_mask  ], axis=(0, 2))
        rf_maps[:, yi, xi] = resp - base

peak = np.array([gaussian_filter(rf_maps[u], RF_SMOOTH).max() for u in range(n_soma)])
print(f"Peak z-score range: [{peak.min():.3f}, {peak.max():.3f}]")

In [ ]:
# ── Figure 1a: Population PSTH ─────────────────────────────────────────
pop = np.nanmean(z, axis=1)           # (trial, time)
m   = np.nanmean(pop, axis=0)
s   = np.nanstd( pop, axis=0) / np.sqrt(np.isfinite(pop).sum(axis=0).clip(1))

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.axvspan(0, 0.25, color="gray", alpha=0.12, label="stim on (250 ms)")
ax.axhline(0, color="gray", lw=0.5, ls="--")
ax.axvline(0, color="gray", lw=0.8, ls=":", label="stimulus onset")
ax.fill_between(tc, m - s, m + s, color="#4878CF", alpha=0.3)
ax.plot(tc, m, color="#4878CF", lw=2)

bl_mu = np.nanmean(m[tc < 0]); bl_sd = np.nanstd(m[tc < 0])
first = np.where((tc > 0) & (m > bl_mu + 2*bl_sd))[0]
if len(first):
    lat = tc[first[0]]
    ax.axvline(lat, color="red", lw=1.2, ls="--")
    ax.text(lat + 0.05, m.max(), f"≈{lat*1000:.0f} ms", color="red", fontsize=9)

ax.set(xlabel="Time from onset (s)", ylabel="Mean z-score",
       title=f"Mesoscope {PLANE} — population PSTH  ({n_soma} soma, {valid.sum()} RF trials)",
       xlim=WINDOW)
ax.legend(fontsize=8, loc="upper right")
plt.tight_layout()
plt.show()

# ── Figure 1b: Top individual RF maps ─────────────────────────────────
order   = np.argsort(peak)[::-1][:N_TOP]
all_sm  = np.array([gaussian_filter(rf_maps[u], RF_SMOOTH) for u in range(n_soma)])
vmax    = np.percentile(np.abs(all_sm[order]), 98)
ext     = [xs_grid[0]-5, xs_grid[-1]+5, ys_grid[0]-5, ys_grid[-1]+5]

n_cols = 4
n_rows = int(np.ceil(N_TOP / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*2.8, n_rows*2.8), squeeze=False)

for plot_i in range(n_rows * n_cols):
    row, col = divmod(plot_i, n_cols)
    ax = axes[row, col]
    if plot_i >= N_TOP:
        ax.set_visible(False); continue
    uid = order[plot_i]
    sm  = gaussian_filter(rf_maps[uid], RF_SMOOTH)
    im  = ax.imshow(sm, origin="lower", cmap="RdBu_r",
                    extent=ext, vmin=-vmax, vmax=vmax, aspect="equal")
    pk  = np.unravel_index(np.argmax(sm), sm.shape)
    ax.plot(xs_grid[pk[1]], ys_grid[pk[0]], "k+", ms=8, mew=1.5)
    ax.set(xticks=xs_grid[::3], yticks=ys_grid[::3])
    ax.tick_params(labelsize=6)
    ax.set_title(f"ROI {uid}  z={peak[uid]:.2f}", fontsize=8, pad=2)
    if col == 0: ax.set_ylabel("Y (°)", fontsize=7)
    if row == n_rows-1: ax.set_xlabel("X (°)", fontsize=7)

cbar_ax = fig.add_axes([0.93, 0.15, 0.015, 0.7])
fig.colorbar(im, cax=cbar_ax, label="z-score vs baseline")
fig.suptitle(f"Mesoscope {PLANE} — top {N_TOP} soma RF maps  (sub-839909)",
             fontsize=10, fontweight="bold")
fig.tight_layout(rect=[0, 0, 0.92, 0.95])
plt.show()

---
## Section 2 — Ecephys Oddball Mismatch Responses

**Dataset:** dandiset 001256 (ecephys, Neuropixels)  
**Pipeline:**
1. Filter to **single units (SUA)** with a clear visual response (RF peak ≥ 5 spk/s above baseline)
2. Align spikes to each trial onset in a ±0.5–1.0 s window
3. Z-score against standard-trial baseline
4. Plot one PSTH line per trial type: standard, halt, omission, orientation deviants

In [ ]:
# ── Config ────────────────────────────────────────────────────────────
EPHYS_ASSET_ID = "cd175e65-8faa-4216-86af-c1fd30e571a1"
RF_WINDOW    = (-0.1,  0.5)   # for visual unit selection
RF_BIN       = 0.01
RESP_WIN     = (0.03, 0.25)   # response window for RF peak
RESP_THRESH  = 5.0            # spk/s — visual unit threshold
RF_SMOOTH_E  = 0.8
ODD_WINDOW   = (-0.5,  1.0)
ODD_BIN      = 0.025
N_STD_MAX    = 300            # cap standard trials to balance sampling

COLORS = {
    "standard":         "#4878CF",
    "halt":             "#D65F5F",
    "omission":         "#E07B39",
    "orientation_45":   "#6BAF6B",
    "orientation_90":   "#9B59B6",
    "sequence_omission":"#C0392B",
}

# ── Load ──────────────────────────────────────────────────────────────
print("Opening ecephys NWB…")
t0 = time.time()
ehandle = open_nwb(EPHYS_ASSET_ID)
trials  = load_trials(ehandle)
print(f"  Opened in {time.time()-t0:.1f}s  |  {len(trials)} total stimulus presentations")

eh5        = ehandle.h5
units_grp  = eh5["units"]
n_total    = int(units_grp["id"].shape[0])
dl         = units_grp["decoder_label"][:]
dl         = np.array([v.decode() if isinstance(v, bytes) else str(v) for v in dl])
sua_idx    = np.where(dl == "sua")[0]
all_spikes = units_grp["spike_times"][:]
spk_index  = units_grp["spike_times_index"][:]

print(f"  {n_total} units total, {len(sua_idx)} SUA")

# ── Helper: bin spikes → (n_trials, n_units, n_bins) firing rates ─────
def bin_spikes(uid_arr, spikes, index, onsets, window, bin_size):
    pre, post = window
    edges   = np.arange(pre, post + bin_size, bin_size)
    centers = 0.5 * (edges[:-1] + edges[1:])
    out = np.zeros((len(onsets), len(uid_arr), len(centers)), dtype=np.float32)
    for j, uid in enumerate(uid_arr):
        i0  = int(index[uid-1]) if uid > 0 else 0
        i1  = int(index[uid])
        spk = spikes[i0:i1]
        for i, t0 in enumerate(onsets):
            rel = spk - t0
            in_w = rel[(rel >= pre) & (rel < post)]
            if len(in_w):
                cnt, _ = np.histogram(in_w, bins=edges)
                out[i, j, :] = cnt / bin_size
    return out, centers

In [ ]:
# ── Step 1: find visual units via RF mapping ──────────────────────────
print("Identifying visual units via RF mapping block…")
rf_t = trials[trials["block_kind"] == "rf_mapping"].reset_index(drop=True)

t1 = time.time()
rf_arr, rf_centers = bin_spikes(sua_idx, all_spikes, spk_index,
                                 rf_t["start_time"].values, RF_WINDOW, RF_BIN)
print(f"  Binned in {time.time()-t1:.1f}s")

xs_e = np.array(sorted(rf_t["x"].unique()))
ys_e = np.array(sorted(rf_t["y"].unique()))
bl_m = rf_centers < 0
rsp  = (rf_centers >= RESP_WIN[0]) & (rf_centers < RESP_WIN[1])
delta = rf_arr[:, :, rsp].mean(axis=2) - rf_arr[:, :, bl_m].mean(axis=2)

rf_maps_e = np.zeros((len(sua_idx), len(ys_e), len(xs_e)), dtype=np.float32)
for xi, x in enumerate(xs_e):
    for yi, y in enumerate(ys_e):
        m = (rf_t["x"].values == x) & (rf_t["y"].values == y)
        rf_maps_e[:, yi, xi] = delta[m, :].mean(axis=0)

peak_e   = np.array([gaussian_filter(rf_maps_e[u], RF_SMOOTH_E).max() for u in range(len(sua_idx))])
vis_mask = peak_e >= RESP_THRESH
vis_idx  = sua_idx[vis_mask]
n_vis    = len(vis_idx)

print(f"\n  Unit summary:")
print(f"    All units : {n_total}")
print(f"    SUA       : {len(sua_idx)}  ({100*len(sua_idx)/n_total:.0f}% of all)")
print(f"    Visual SUA: {n_vis}  ({100*n_vis/len(sua_idx):.0f}% of SUA)")

# ── Step 2: select oddball block, split by trial type ─────────────────
odd_t      = trials[trials["block_kind"] == "paradigm_oddball"]
first_blk  = odd_t["block"].unique()[0]
ob         = odd_t[odd_t["block"] == first_blk]
dev_types  = sorted(ob[ob["is_deviant"]]["trial_type"].unique().tolist())
trial_types = ["standard"] + dev_types
print(f"\n  Trial types in '{first_blk}': {trial_types}")

type_trials = {}
for tt in trial_types:
    sub = ob[ob["trial_type"] == tt]
    if tt == "standard" and len(sub) > N_STD_MAX:
        sub = sub.sample(N_STD_MAX, random_state=42)
    type_trials[tt] = sub.reset_index(drop=True)
    print(f"    {tt:20s}: {len(type_trials[tt])} trials")

# ── Step 3: bin oddball spikes per type ───────────────────────────────
print("\nBinning oddball spikes…")
type_arrays = {}
t1 = time.time()
for tt, sub in type_trials.items():
    arr, centers = bin_spikes(vis_idx, all_spikes, spk_index,
                               sub["start_time"].values, ODD_WINDOW, ODD_BIN)
    type_arrays[tt] = arr
print(f"  Done in {time.time()-t1:.1f}s")

# Z-score: baseline from standard-trial distribution
std_arr = type_arrays["standard"]
bl_t    = (centers >= -0.5) & (centers < 0.0)
mu_z    = std_arr[:, :, bl_t].mean(axis=(0, 2), keepdims=True)
sig_z   = std_arr[:, :, bl_t].std(axis=(0, 2), keepdims=True)
sig_z   = np.where(sig_z > 0.1, sig_z, 1.0)
type_z  = {tt: (arr - mu_z) / sig_z for tt, arr in type_arrays.items()}
print("Z-scored.")

In [ ]:
# ── Figure 2: Oddball PSTH split by deviant type ─────────────────────
def mean_sem(arr):
    per_trial = arr.mean(axis=1)   # (trial, time)
    return per_trial.mean(axis=0), per_trial.std(axis=0) / np.sqrt(per_trial.shape[0])

fig, ax = plt.subplots(figsize=(9, 5))

ax.axvspan(0, 0.25, color="gray", alpha=0.10, label="stim (250 ms)")
ax.axhline(0,    color="gray", lw=0.6, ls="--")
ax.axvline(0,    color="gray", lw=0.8, ls=":")
ax.axvline(0.25, color="gray", lw=0.6, ls=":")

# Standard (background reference)
m, s = mean_sem(type_z["standard"])
ax.fill_between(centers, m-s, m+s, color=COLORS["standard"], alpha=0.20)
ax.plot(centers, m, color=COLORS["standard"], lw=2.5,
        label=f"standard  (n={len(type_trials['standard'])})", zorder=3)

# Deviant types
for tt in dev_types:
    color = COLORS.get(tt, "#888888")
    m, s  = mean_sem(type_z[tt])
    n     = len(type_trials[tt])
    ax.fill_between(centers, m-s, m+s, color=color, alpha=0.18)
    ax.plot(centers, m, color=color, lw=2, label=f"{tt}  (n={n})", zorder=4)

ax.set_xlabel("Time from stimulus onset (s)", fontsize=11)
ax.set_ylabel("Population response (z-score)", fontsize=11)
ax.set_title(
    f"Ecephys oddball PSTH — {first_blk} — deviant types\n"
    f"Visual SUA: {n_vis}/{len(sua_idx)} ({100*n_vis/len(sua_idx):.0f}% of SUA, "
    f"{100*n_vis/n_total:.0f}% of all units)",
    fontsize=10
)
ax.set_xlim(ODD_WINDOW)
ax.legend(fontsize=9, framealpha=0.8, loc="upper right")
plt.tight_layout()
plt.show()

ehandle.close()
print("Done.")